# Evidence: Outlook emails (.msg) for the AI

**What it does**
Converts Outlook emails (`.msg`) in the evidences folder into files the AI can read:
- `<name>.msg.md`: the email as text. It has a header table (From, To, Cc, date, subject, number of earlier emails), then the body. The body is taken from the HTML version when there is one, so lists, tables, red text, links as `text (url)` and the place of inline images (`[inline image: ...]`) are kept. Each earlier email quoted in the thread becomes its own section (`### Earlier email 2 of 5`), so the AI knows who wrote what and when. A list of attachments follows, saying what each one became.
- `<name>.msg_attachments/`: every attachment, saved and converted by the matching service:
  - Word becomes a PDF (`DocxConverter`)
  - Excel becomes text and pictures (`ExcelConverter`)
  - an email attached to the email is converted the same way, up to 3 levels deep
  - PDFs, images and text files are kept as they are

**Why**
A lot of evidence arrives as emails: approvals, confirmations, forwarded reports. Gemini and Claude cannot open `.msg`, Outlook's own format. The logic is in the `MsgConverter` service (`services/evidences/msg_converter.py`), so the API can use the same code later.

**How to use**
1. Put `.msg` files in the evidences folder (`storage/evidences/` or `KCT_EVIDENCES_ROOT`). In Outlook: drag an email to a folder, or File > Save As > Outlook Message Format. No file yet? Run Step 1 to create a fake one.
2. Run all cells. To convert one email, put its name in `FILE` (Step 3).
3. Read Step 4: that text is exactly what the AI gets for the email.

**Limits**
- Attachments in other formats (`.zip`, `.doc`, `.xls`, `.pptx`) are listed but not sent: the models cannot read them.
- Encrypted or signed emails (S/MIME) may not open.

**Needs**: `extract-msg` (`pip install extract-msg`, or Step 3 of `01_setup.ipynb`). It is GPL-3.0 licensed: fine for internal use, but have it checked if ING has rules on GPL packages.

## Setup

Finds the project folder, reads `.env`, and shows the evidences folder.

In [ ]:
import os, sys
from pathlib import Path

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from services.app import boot

kct = boot()
evidences = kct.evidences
print("Evidences folder:", evidences.root)

## Step 1: Fake evidence email (optional)

Creates `EV-04_evidence_email_SAMPLE.msg` with `MsgFactory` (`factories/msg_factory.py`). It is the newest reply in an evidence thread between IAM Operations, the auditor, the approver and the service desk:
- To and Cc recipients, a plain-text and an HTML body with a list, red text and links
- `HISTORY` earlier emails quoted below it, each with Outlook's From / Sent / To / Subject block
- `IMAGES` inline screenshots (audit log, ServiceNow tickets), the first in the newest email and the rest spread over the earlier ones, named `image001.png`, `image002.png`, ... as Outlook does
- `LINKS`: links to ServiceNow tickets, the audit portal and SharePoint
- `ATTACHMENTS`: the Word evidence (`DocxFactory`), an Excel extract (`ExcelFactory`, 15 rows) and an approval email forwarded as an attachment

Running it again overwrites the file.

In [ ]:
from factories import MsgFactory

HISTORY = 4        # earlier emails quoted below the newest one (0 = a single email)
IMAGES = 3         # inline screenshots spread over the thread
LINKS = True       # ServiceNow, audit portal and SharePoint links
ATTACHMENTS = True # Word, Excel and a forwarded approval email

factory = MsgFactory(history=HISTORY, images=IMAGES, links=LINKS, attachments=ATTACHMENTS)
print("Created", factory.create(evidences.root))

## Step 2: Emails in the evidences folder

Every `.msg`, with its size. "needs conversion" means there is no `.md` yet, or the email file changed after it was made. Emails inside `*.msg_attachments` folders are converted with their parent email and are not listed.

In [ ]:
emails = [p for p in evidences.files("*.msg")
          if not any(part.endswith(".msg_attachments") for part in p.parts[:-1])]
if not emails:
    print("No .msg files in", evidences.root, "- add some or run Step 1")
for relative in emails:
    path = evidences.path(relative)
    text = path.with_name(path.name + ".md")
    state = "converted" if text.is_file() and text.stat().st_mtime >= path.stat().st_mtime else "needs conversion"
    print(f"{str(relative):<60} {path.stat().st_size / 1024:>8.1f} KB  {state}")

## Step 3: Convert one email

Set `FILE` to a name from the list above. Prints each attachment and what it became.

In [ ]:
FILE = "EV-04_evidence_email_SAMPLE.msg"

try:
    result = evidences.convert_email(FILE)
except Exception as exc:
    print("❌", type(exc).__name__, exc)
else:
    print(f"✅ {result.text.name}  (about {result.estimated_tokens:,} tokens)  subject: {result.subject}")
    for attachment in result.attachments:
        outputs = ", ".join(p.name for p in attachment.outputs) or "-"
        note = f"  ({attachment.note})" if attachment.note else ""
        print(f"   {attachment.kind:<6} {attachment.name}  ->  {outputs}{note}")
    print(f"\n{len(result.files)} file(s) for the AI in total")

## Step 4: What the AI will read

The email text made in Step 3: header, body and the list of attachments.

In [ ]:
print(result.text.read_text(encoding="utf-8"))

## Step 5: Convert everything that needs it

Converts every email (and Word and Excel file) in the folder whose output is missing or older than the file. `FORCE = True` converts all of them again.

In [ ]:
FORCE = False

converted = evidences.convert_all(force=FORCE)
if not converted:
    print("✅ Everything is already converted")
for item in converted:
    print("✅", item.output.relative_to(evidences.root), f"({item.engine})")

## Step 6: Ready for the AI

Every file in the evidences folder the AI can read, including what came out of the emails' attachments. The assessment will send these with the test steps.

In [ ]:
for doc in evidences.attachments():
    print(f"{doc.title[:90]:<90} {len(doc.data) / 1024:>8.1f} KB  {doc.mime_type}")